# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohidraheel/Machine-Learning-Practice/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb huggingface_hub pandas numpy

In [2]:
import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import login
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

print("Libraries loaded.")

Libraries loaded.


In [3]:
hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found. Add it in Colab Secrets and enable notebook access."
    )

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)

print("Hugging Face authentication completed.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face authentication completed.


In [4]:
con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("SET secret_directory='/tmp'")

con.execute(f'''
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    )
''')

WAREHOUSE_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

con.execute(f'''
    CREATE OR REPLACE VIEW warehouse AS
    SELECT *
    FROM read_parquet(
        '{WAREHOUSE_PATH}',
        hive_partitioning=true
    )
''')

print("Warehouse view created.")

Warehouse view created.


In [5]:
schema_df = con.execute("DESCRIBE warehouse").df()
display(schema_df)

required_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_data_available",
    "ga4_sessions",
]

available_columns = schema_df["column_name"].tolist()

missing_columns = [
    column for column in required_columns
    if column not in available_columns
]

if missing_columns:
    raise KeyError(f"Missing required warehouse columns: {missing_columns}")

print("Required warehouse columns are available.")

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


Required warehouse columns are available.


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [6]:
audit_sql = '''
SELECT
    client_hash_id AS client_id,
    content_hash_id AS content_id,

    SUM(COALESCE(gsc_impressions, 0)) AS impressions_31d,
    SUM(COALESCE(gsc_clicks, 0)) AS clicks_31d,

    CASE
        WHEN SUM(COALESCE(gsc_impressions, 0)) > 0
        THEN SUM(COALESCE(gsc_clicks, 0)) * 1.0
             / SUM(COALESCE(gsc_impressions, 0))
        ELSE NULL
    END AS ctr_31d,

    CASE
        WHEN SUM(
            CASE
                WHEN gsc_avg_position IS NOT NULL
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) > 0
        THEN SUM(
            CASE
                WHEN gsc_avg_position IS NOT NULL
                THEN gsc_avg_position * COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) * 1.0
        / SUM(
            CASE
                WHEN gsc_avg_position IS NOT NULL
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        )
        ELSE NULL
    END AS weighted_position_31d,

    SUM(COALESCE(ga4_sessions, 0)) AS sessions_31d,
    COUNT(*) AS source_rows

FROM warehouse

WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
  AND gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id

HAVING SUM(COALESCE(gsc_impressions, 0)) >= 100
'''

audit_frame = con.execute(audit_sql).df()

print("Eligible rows:", len(audit_frame))
display(audit_frame.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible rows: 101441


,client_id,content_id,impressions_31d,clicks_31d,ctr_31d,weighted_position_31d,sessions_31d,source_rows
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,0.001754,4.450877,0.0,31
1,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,0.000000,5.637584,4.0,30
2,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,0.004222,6.906404,9.0,31
3,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,0.005776,3.950542,3.0,31
4,client_73cda7b4e4f265ea,content_2662845f598544ef,150.0,1.0,0.006667,7.506667,1.0,30
5,client_73cda7b4e4f265ea,content_712c365258cee05c,6048.0,23.0,0.003803,4.931878,8.0,31
6,client_73cda7b4e4f265ea,content_476c37c366920c1b,223.0,0.0,0.000000,61.538117,1.0,30
7,client_73cda7b4e4f265ea,content_3dba50ae010f3f30,357.0,1.0,0.002801,18.876751,0.0,30
8,client_73cda7b4e4f265ea,content_d720dde3701523c0,132.0,1.0,0.007576,30.863636,0.0,29
9,client_73cda7b4e4f265ea,content_098eedbd77ec1de1,281.0,1.0,0.003559,38.199288,3.0,30


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal 1 — CTR versus search position

Assumption: pages ranking nearer the top should generally have higher observed CTR.

### Signal 2 — Impression volume

Assumption: impression volume separates small observed opportunities from large observed opportunities.

### Signal 3 — Search clicks versus sessions

Assumption: pages with more measured search clicks should generally also show more measured sessions, while attribution differences may weaken the relationship.

In [7]:
position_bins = [0, 3, 5, 10, 20, 50, np.inf]
position_labels = ["1-3", "4-5", "6-10", "11-20", "21-50", "51+"]

position_test = audit_frame.dropna(
    subset=["weighted_position_31d", "ctr_31d"]
).copy()

position_test["position_bucket"] = pd.cut(
    position_test["weighted_position_31d"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True,
)

position_table = (
    position_test
    .groupby("position_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        median_ctr=("ctr_31d", "median"),
        mean_ctr=("ctr_31d", "mean"),
    )
    .reset_index()
)

position_table["median_ctr_pct"] = (
    100 * position_table["median_ctr"]
).round(2)

position_table["mean_ctr_pct"] = (
    100 * position_table["mean_ctr"]
).round(2)

display(
    position_table[
        ["position_bucket", "n", "median_ctr_pct", "mean_ctr_pct"]
    ]
)

medians = position_table["median_ctr"].dropna().tolist()

declining_steps = sum(
    later <= earlier
    for earlier, later in zip(medians, medians[1:])
)

if len(medians) >= 2 and declining_steps == len(medians) - 1:
    signal_1_verdict = "CONFIRMED"
elif len(medians) >= 2 and declining_steps == 0:
    signal_1_verdict = "OPPOSITE"
elif declining_steps > 0:
    signal_1_verdict = "MIXED"
else:
    signal_1_verdict = "FALSE"

print("Signal 1 verdict:", signal_1_verdict)

,position_bucket,n,median_ctr_pct,mean_ctr_pct
0,1-3,10194,0.21,0.34
1,4-5,17167,0.23,0.36
2,6-10,30644,0.16,0.30
3,11-20,19547,0.10,0.24
4,21-50,19758,0.00,0.14
5,51+,4131,0.00,0.05


Signal 1 verdict: MIXED


In [9]:
volume_test = audit_frame.dropna(
    subset=["impressions_31d", "ctr_31d"]
).copy()

volume_test["volume_bucket"] = pd.qcut(
    volume_test["impressions_31d"],
    q=5,
    labels=["Very low", "Low", "Medium", "High", "Very high"],
    duplicates="drop",
)

volume_table = (
    volume_test
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        minimum_impressions=("impressions_31d", "min"),
        median_impressions=("impressions_31d", "median"),
        maximum_impressions=("impressions_31d", "max"),
        median_ctr=("ctr_31d", "median"),
    )
    .reset_index()
)

volume_table["median_ctr_pct"] = (
    100 * volume_table["median_ctr"]
).round(2)

display(volume_table)

volume_medians = volume_table["median_impressions"].dropna().tolist()

signal_2_verdict = (
    "CONFIRMED"
    if all(
        later > earlier
        for earlier, later in zip(volume_medians, volume_medians[1:])
    )
    else "FALSE"
)

print("Signal 2 verdict:", signal_2_verdict)
print(
    "Interpretation: volume separates opportunity size; "
    "it does not prove that volume causes low CTR."
)

,volume_bucket,n,minimum_impressions,median_impressions,maximum_impressions,median_ctr,median_ctr_pct
0,Very low,20365,100.0,154.0,231.0,0.000000,0.00
1,Low,20238,232.0,347.0,520.0,0.000000,0.00
2,Medium,20272,521.0,787.0,1215.0,0.001456,0.15
3,High,20278,1216.0,1946.0,3348.0,0.001823,0.18
4,Very high,20288,3349.0,6416.5,617124.0,0.002147,0.21


Signal 2 verdict: CONFIRMED
Interpretation: volume separates opportunity size; it does not prove that volume causes low CTR.


In [10]:
session_click_frame = audit_frame.dropna(
    subset=["clicks_31d", "sessions_31d"]
).copy()

session_click_frame["click_bucket"] = pd.qcut(
    session_click_frame["clicks_31d"].rank(method="first"),
    q=5,
    labels=["Very low", "Low", "Medium", "High", "Very high"],
)

session_click_table = (
    session_click_frame
    .groupby("click_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        median_clicks=("clicks_31d", "median"),
        median_sessions=("sessions_31d", "median"),
        mean_sessions=("sessions_31d", "mean"),
    )
    .reset_index()
)

display(session_click_table)

spearman_value = session_click_frame[
    ["clicks_31d", "sessions_31d"]
].corr(method="spearman").iloc[0, 1]

if pd.isna(spearman_value) or abs(spearman_value) < 0.10:
    signal_3_verdict = "FALSE"
elif spearman_value < 0:
    signal_3_verdict = "OPPOSITE"
elif spearman_value >= 0.50:
    signal_3_verdict = "CONFIRMED"
else:
    signal_3_verdict = "MIXED"

print("Spearman correlation:", round(float(spearman_value), 3))
print("Signal 3 verdict:", signal_3_verdict)

,click_bucket,n,median_clicks,median_sessions,mean_sessions
0,Very low,20289,0.0,0.0,5.316773
1,Low,20288,0.0,0.0,2.017399
2,Medium,20288,1.0,1.0,6.987924
3,High,20288,4.0,3.0,11.351538
4,Very high,20288,19.0,12.0,34.487037


Spearman correlation: 0.473
Signal 3 verdict: MIXED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

This section audits the assumption behind FlyRank's CTR-fix logic:

> CTR should be judged relative to search position rather than by one universal threshold.

Each page is compared with the median CTR of its own position bucket.

In [11]:
expected_ctr_table = (
    position_test
    .groupby("position_bucket", observed=True)
    .agg(
        expected_ctr=("ctr_31d", "median"),
        bucket_n=("content_id", "size"),
    )
    .reset_index()
)

flag_test = position_test.merge(
    expected_ctr_table,
    on="position_bucket",
    how="left",
)

flag_test["ctr_gap"] = (
    flag_test["expected_ctr"] - flag_test["ctr_31d"]
)

flag_test["below_bucket_median"] = (
    flag_test["ctr_gap"] > 0
)

flag_summary = (
    flag_test
    .groupby("position_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        expected_ctr=("expected_ctr", "first"),
        below_expected_n=("below_bucket_median", "sum"),
        median_absolute_gap=("ctr_gap", lambda s: s.abs().median()),
    )
    .reset_index()
)

flag_summary["expected_ctr_pct"] = (
    100 * flag_summary["expected_ctr"]
).round(2)

flag_summary["below_expected_rate_pct"] = (
    100 * flag_summary["below_expected_n"] / flag_summary["n"]
).round(2)

flag_summary["median_absolute_gap_pct_points"] = (
    100 * flag_summary["median_absolute_gap"]
).round(2)

display(
    flag_summary[
        [
            "position_bucket",
            "n",
            "expected_ctr_pct",
            "below_expected_n",
            "below_expected_rate_pct",
            "median_absolute_gap_pct_points",
        ]
    ]
)

print("Flag-linked verdict:", signal_1_verdict)

,position_bucket,n,expected_ctr_pct,below_expected_n,below_expected_rate_pct,median_absolute_gap_pct_points
0,1-3,10194,0.21,5095,49.98,0.21
1,4-5,17167,0.23,8582,49.99,0.19
2,6-10,30644,0.16,15319,49.99,0.16
3,11-20,19547,0.10,9772,49.99,0.10
4,21-50,19758,0.00,0,0.00,0.00
5,51+,4131,0.00,0,0.00,0.00


Flag-linked verdict: MIXED


In [12]:
universal_median_ctr = float(flag_test["ctr_31d"].median())

flag_test["universal_gap"] = (
    universal_median_ctr - flag_test["ctr_31d"]
)

flag_test["bucket_gap"] = (
    flag_test["expected_ctr"] - flag_test["ctr_31d"]
)

comparison_table = (
    flag_test
    .groupby("position_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        universal_median_gap=("universal_gap", "median"),
        bucket_median_gap=("bucket_gap", "median"),
    )
    .reset_index()
)

comparison_table["universal_median_gap_pct_points"] = (
    100 * comparison_table["universal_median_gap"]
).round(2)

comparison_table["bucket_median_gap_pct_points"] = (
    100 * comparison_table["bucket_median_gap"]
).round(2)

display(comparison_table)

,position_bucket,n,universal_median_gap,bucket_median_gap,universal_median_gap_pct_points,bucket_median_gap_pct_points
0,1-3,10194,-0.000877,0.0,-0.09,0.0
1,4-5,17167,-0.001072,0.0,-0.11,0.0
2,6-10,30644,-0.000386,0.0,-0.04,0.0
3,11-20,19547,0.000230,0.0,0.02,0.0
4,21-50,19758,0.001238,0.0,0.12,0.0
5,51+,4131,0.001238,0.0,0.12,0.0


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*


The observed data supports interpreting CTR relative to search position.

Impression volume can prioritize larger measured opportunities, but it does not prove that a page needs editing. Search intent, SERP features, seasonality, query mix, and business context still require human review.

In [13]:
verdicts = pd.DataFrame([
    {
        "signal": "CTR versus search position",
        "verdict": signal_1_verdict,
        "practical_use": "Use position-relative CTR expectations.",
    },
    {
        "signal": "Impression volume",
        "verdict": signal_2_verdict,
        "practical_use": "Use volume to prioritize opportunity size.",
    },
    {
        "signal": "Search clicks versus sessions",
        "verdict": signal_3_verdict,
        "practical_use": "Use as supporting context, not an identity.",
    },
    {
        "signal": "FlyRank CTR-fix assumption",
        "verdict": signal_1_verdict,
        "practical_use": "Prefer bucket-relative CTR over one universal threshold.",
    },
])

display(verdicts)

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

audit_receipt = {
    "assignment": "Week 4 signal audit",
    "audit_period": "2026-03",
    "eligible_rows": int(len(audit_frame)),
    "signal_1_verdict": signal_1_verdict,
    "signal_2_verdict": signal_2_verdict,
    "signal_3_verdict": signal_3_verdict,
    "flag_linked_verdict": signal_1_verdict,
    "spearman_clicks_sessions": (
        None if pd.isna(spearman_value)
        else round(float(spearman_value), 6)
    ),
    "future_inputs_used": False,
    "product_flags_used": False,
    "private_fields_used": False,
}

audit_receipt_path = OUTPUT_DIR / "w04_signal_audit_metrics.json"

with open(audit_receipt_path, "w", encoding="utf-8") as file:
    json.dump(audit_receipt, file, indent=2)

print("Audit receipt written to:", audit_receipt_path)
display(pd.Series(audit_receipt, name="value").to_frame())

,signal,verdict,practical_use
0,CTR versus search position,MIXED,Use position-relative CTR expectations.
1,Impression volume,CONFIRMED,Use volume to prioritize opportunity size.
2,Search clicks versus sessions,MIXED,"Use as supporting context, not an identity."
3,FlyRank CTR-fix assumption,MIXED,Prefer bucket-relative CTR over one universal threshold.


Audit receipt written to: work/outputs/w04_signal_audit_metrics.json


,value
assignment,Week 4 signal audit
audit_period,2026-03
eligible_rows,101441
signal_1_verdict,MIXED
signal_2_verdict,CONFIRMED
signal_3_verdict,MIXED
flag_linked_verdict,MIXED
spearman_clicks_sessions,0.473188
future_inputs_used,False
product_flags_used,False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [14]:
valid_verdicts = {"CONFIRMED", "OPPOSITE", "MIXED", "FALSE"}

assert signal_1_verdict in valid_verdicts
assert signal_2_verdict in valid_verdicts
assert signal_3_verdict in valid_verdicts
assert len(audit_frame) > 0
assert audit_receipt_path.exists()

print("SIGNAL AUDIT COMPLETE")
print("Eligible rows:", len(audit_frame))
print("Receipt:", audit_receipt_path)

SIGNAL AUDIT COMPLETE
Eligible rows: 101441
Receipt: work/outputs/w04_signal_audit_metrics.json
